# CIC-IDS2017 Dataset Exploration

This notebook orchestrates the project's Phase 02 ingestion utilities and reusable Phase 03 EDA functions. Raw CSVs are read only. Full-data aggregates are exact and chunked; expensive distributions, correlations, and duplicate diagnostics use documented deterministic samples.

The latest executed scan is recorded in `reports/eda/CIC-IDS2017_eda.md` and JSON. Re-run Section 3 to refresh the analysis and figures.


## 1. Project and dataset introduction

CIC-IDS2017 is a labelled network-flow benchmark published by the Canadian Institute for Cybersecurity at the University of New Brunswick. This analysis uses the extracted `MachineLearningCVE` CSVs, which contain CICFlowMeter-derived features and original labels. The [official dataset page](https://www.unb.ca/cic/datasets/ids-2017.html) describes a controlled five-day July 2017 collection and provides additional material such as PCAPs and profiles.

The analysis investigates composition, labels, imbalance, data quality, feature distributions and relationships, per-file patterns, duplicates, and leakage risks. The benchmark is not a sample of current operational traffic or real-world attack prevalence.


## 2. Environment and imports

Run from the project root in an environment installed with `requirements.txt`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

from intrusion_intelligence.config import load_config
from intrusion_intelligence.eda import run_eda
from intrusion_intelligence.ingestion import preview_dataset_file

config = load_config()
figure_dir = config.paths.report_figures / 'eda'


## 3. Dataset discovery and loading

The analysis validates files and schemas through Phase 02 ingestion, then reads them sequentially in chunks. Full-scan results are separated from a uniform sample for population plots/correlations and a capped stratified sample for per-label feature comparisons.

In [ ]:
result, outputs = run_eda(
    config,
    chunk_size=config.ingestion.chunk_size,
    sample_size=20_000,
    max_per_label=1_000,
    seed=config.experiment.random_seed,
)
summary = result.summary
display(Markdown(
    f"Full scan: {summary['dataset_overview']['rows_full']:,} rows in "
    f"{summary['dataset_overview']['files_count']} files; uniform sample: "
    f"{len(result.uniform_sample):,} rows, seed {summary['analysis_method']['sample_seed']}."
))
display(pd.DataFrame(summary['dataset_overview']['files']))
print(outputs.markdown_report)


## 4. Dataset overview

Dimensions and per-file row counts are exact. Dtype profiles reflect chunk-level CSV inference; numeric classification is based on successful numeric parsing and does not coerce source records.

In [ ]:
overview = summary['dataset_overview']
display(pd.DataFrame([{key: overview[key] for key in (
    'rows_full', 'files_count', 'columns_count', 'feature_columns_count',
    'numeric_columns_count', 'non_numeric_or_mixed_columns_count',
    'rows_with_missing_full', 'sample_rows_uniform', 'sample_seed'
)}]).T.rename(columns={0: 'value'}))
display(pd.DataFrame(overview['columns']))


## 5. Initial data inspection

The preview is bounded and read-only. The uniform sample is reproducible; representative preview values retain original labels and source-column aliases.

In [ ]:
first_file = config.dataset.directory / overview['files'][0]['file']
preview = preview_dataset_file(
    first_file,
    label_candidates=config.dataset.label_candidates,
    encoding=config.ingestion.csv_encoding,
    row_count=5,
)
display(preview)
display(result.uniform_sample.sample(
    min(5, len(result.uniform_sample)),
    random_state=config.experiment.random_seed,
))
display(pd.DataFrame({'dtype': result.uniform_sample.dtypes.astype(str)}))


## 6. Label distribution analysis

Counts and percentages use all scanned rows and preserve exact source labels. Benign/malicious is a derived view only; source values are not merged or changed.

In [ ]:
labels = summary['label_distribution']
display(pd.DataFrame({
    'rows_full': labels['counts'],
    'percent_full': labels['percentages'],
}).sort_values('rows_full', ascending=False))
display(Image(filename=str(figure_dir / 'label_distribution.png')))
display(Image(filename=str(figure_dir / 'benign_malicious_distribution.png')))
display(Image(filename=str(figure_dir / 'per_file_label_distribution.png')))


## 7. Class imbalance investigation

The majority/minority ratio and per-file support inform later metric and split design. No balancing or resampling is performed.

In [ ]:
imbalance = labels['class_imbalance']
display(pd.DataFrame([{
    key: imbalance[key] for key in (
        'majority_label', 'majority_count', 'minority_label', 'minority_count',
        'majority_to_minority_ratio', 'categories_below_100_rows',
        'non_benign_categories_in_one_file'
    )
}]).T.rename(columns={0: 'value'}))
display(pd.DataFrame(imbalance['per_label_file_support']).T)


## 8. Missing values investigation

Empty fields and common textual NA tokens are counted for diagnostics only. Nothing is imputed or deleted.

In [ ]:
missing = pd.DataFrame(summary['data_quality']['missing_by_column_full'])
display(missing.sort_values('count_full', ascending=False).head(20))
display(Image(filename=str(figure_dir / 'missing_values.png')))


## 9. Infinite and invalid numerical values

The full scan counts infinities, parse failures, and negative numeric values. Negative values are flagged for investigation, not automatically classified as errors.

In [ ]:
quality = summary['data_quality']
display(pd.DataFrame(quality['numeric_value_issues_full']))
display(pd.DataFrame(quality['negative_numeric_features_full']).head(20))


## 10. Numerical feature statistics

Full-scan moments, minima/maxima, zero shares, and invalid counts are exact. Quantiles, skewness, kurtosis, and cardinality are uniform-sample statistics.

In [ ]:
numeric_stats = pd.DataFrame(summary['numeric_statistics'])
display(numeric_stats.sort_values(
    'skew_sample', key=lambda values: values.abs(), ascending=False
).head(20))


## 11. Feature distribution visualizations

A small set of flow-relevant features is plotted. Any signed-log transformation is for display only; source values are unchanged.

In [ ]:
display(Image(filename=str(figure_dir / 'numeric_feature_distributions.png')))
display(Image(filename=str(figure_dir / 'feature_quality_summary.png')))


## 12. Correlation analysis

Pearson and Spearman correlations use the deterministic uniform sample after masking infinities. The heatmap uses a legible feature subset; the report lists high-correlation pairs. Correlation is not causality or proof of leakage, and no columns are removed.

In [ ]:
correlations = summary['correlations']
display(pd.DataFrame(correlations['pearson_high_pairs']))
display(pd.DataFrame(correlations['spearman_high_pairs']))
display(Image(filename=str(figure_dir / 'correlation_heatmap.png')))


## 13. Feature relationships with attack labels

Within-label comparisons use a deterministic sample capped per category. They describe feature shapes and are not class-prevalence estimates or causal claims.

In [ ]:
relationships = summary['feature_relationships']
display(Markdown(relationships['interpretation_note']))
for feature, by_label in relationships['statistics'].items():
    display(Markdown(f'### {feature}'))
    display(pd.DataFrame(by_label).T)
display(Image(filename=str(figure_dir / 'feature_label_comparisons.png')))


## 14. Network protocol and port analysis

Destination-port counts are exact. This extracted schema has no standalone protocol or source-port field. Port patterns can reflect the controlled scenarios in this benchmark.

In [ ]:
network_fields = summary['network_fields']
display(pd.DataFrame({'available': network_fields['presence']}))
display(pd.DataFrame(network_fields['destination_port']['top_ports_full']))
display(Image(filename=str(figure_dir / 'destination_port_distribution.png')))


## 15. Temporal and per-file analysis

No per-flow timestamp exists in this schema, so exact time ranges and interval trends cannot be calculated. Per-file rows and label composition are exact.

In [ ]:
display(Markdown(summary['temporal_analysis']['reason']))
display(pd.DataFrame(overview['files'])[
    ['file', 'rows_full', 'columns', 'label_counts_full']
])


## 16. Duplicate and redundancy investigation

Duplicate diagnostics are estimates from the deterministic uniform sample and do not modify or remove rows.

In [ ]:
display(pd.DataFrame([summary['duplicate_analysis']]).T.rename(columns={0: 'value'}))
display(Image(filename=str(figure_dir / 'duplicate_sample_summary.png')))


## 17. Feature quality and potentially uninformative features

Exact constants use the full scan; dominant-value and high-cardinality flags are sample-based. These are candidates for later review, not removal decisions.

In [ ]:
display(Markdown('Exact constants: ' + ', '.join(
    summary['feature_quality']['constant_features_exact_full'])))
display(pd.DataFrame(
    summary['feature_quality']['potentially_uninformative_or_high_cardinality_sample']
))


## 18. Leakage and dataset bias investigation

The extracted schema has no Flow ID, IP, timestamp, protocol, or standalone source-port fields. Destination port is present. Source filename is retained as provenance; the full-scan per-file label support shows whether file/scenario identity tracks labels. Do not add file identity as a predictive feature without a leakage review. No split or feature removal is performed.

In [ ]:
display(pd.DataFrame(imbalance['per_label_file_support']).T)
display(pd.DataFrame({'field_present': network_fields['presence']}))
display(Markdown('Source-file identity remains EDA provenance only.'))


## 19. Key EDA findings

These findings come from the actual scan and distinguish complete counts from sample-based results.

In [ ]:
for finding in summary['key_findings']:
    display(Markdown(f'- {finding}'))


## 20. Recommendations for later phases

### Phase 04 — Data Cleaning & Leakage Prevention

Review the exact missing, infinity, parse-failure, negative-value, and constant-feature evidence in the generated report. Investigate sample duplicate groups with a full external-memory method before deciding whether records should be excluded. Keep source-file provenance out of predictive features until scenario leakage is tested. Preserve raw CSV files.

### Phase 05 — Feature Engineering & Preprocessing

Evaluate robust scaling for heavy-tailed flow measures and investigate flagged redundancy. Plan per-class metrics around the observed imbalance and file-specific category support. Do not treat correlations as automatic feature-removal instructions.
